# Synaptische Routing-Architektur (SRA)
## 11: Dynamische Löschung von Synapsen (Abbruch von Hot-Swap/Bereinigung einer bestimmten Domäne)

In diesem Notizbuch demonstrieren wir eine Funktion von SRA: „synaptische Löschung“. Konkret werden wir die folgenden zwei Szenarien untersuchen.
1. **Plugin entfernen (`pop_synapses`)**: Später hinzugefügte Synapsen (Hot-Swap) physisch vom Ende entfernen und in den Zustand vor dem Hinzufügen zurückversetzen.
2. **Spezifische Domänen bereinigen (`clear_synapses`)**: Nur bestimmte Funktionen (z. B. Mathematik) aus dem anfänglichen Basismodell extrahieren und Synapsen, die nicht mit anderen geteilt werden, sicher auf Null löschen.

*Dieses Notebook kann ohne GPU betrieben werden.

In [ ]:
import sys
import os
import torch
import tiktoken

# SRAライブラリへのパスを通す
sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

from sra_language_models import MoESRALanguageModel
from sra_experiment import find_unshared_synapses

### 1. Plugins entfernen („pop_synapses“)
Lassen Sie uns zunächst ein kleines Modell erstellen, Synapsen dynamisch hinzufügen und sie dann mit „pop_synapses“ entfernen.

In [ ]:
# モデルの初期化
dim = 128
layers = 2
num_synapses = 4
k = 2
syn_hidden = 256
vocab_size = 1000

print("=== プラグインの取り外しデモ ===")
model = MoESRALanguageModel(vocab_size, dim, layers, num_synapses, k, syn_hidden)
print(f"[初期状態] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの追加 (Hot-Swap)
print("\nプラグインとしてシナプスを 2つ 追加します...")
model.add_synapses(2, freeze_base=True)
print(f"[追加後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの取り外し (Undo Hot-Swap)
print("\n追加したシナプスを 1つ 取り外します...")
model.pop_synapses(1)
print(f"[取り外し後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

Auf diese Weise werden durch den Aufruf von „pop_synapses(N)“ die N synaptischen Tensoren physisch vom Ende entfernt und die Kapazität des Modells wiederhergestellt.

### 2. Bereinigen bestimmter Domänen („clear_synapses“ und „find_unshared_synapses“)
Als Nächstes demonstrieren wir den Prozess der automatischen Erkennung „unnötiger Synapsen, die nur in einer bestimmten Domäne (in diesem Fall „Mathematik“) verwendet werden“ aus dem Basismodell, das mehrere Domänen gelernt hat, und der sicheren Deaktivierung dieser Synapsen durch Nulllöschen.

In [ ]:
import random
tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = tokenizer.n_vocab

device = "cuda" if torch.cuda.is_available() else "cpu"

# 仮のデータセットとバッチ関数の準備（デモ用）
domains = ["text", "code", "math"]
data = {d: torch.randint(0, vocab_size, (1000,)) for d in domains}

def dummy_get_batch(data_dict, batch_size, seq_len, domain):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    d_data = data_dict[domain]
    for i in range(batch_size):
        start = random.randint(0, len(d_data) - seq_len - 1)
        x[i] = d_data[start:start+seq_len]
        y[i] = d_data[start+1:start+seq_len+1]
    return x, y

# もう少し大きめのモデルを準備
multi_model = MoESRALanguageModel(vocab_size, dim=128, layers=2, num_synapses=16, k=2, syn_hidden=256).to(device)

In [ ]:
print("=== 特定ドメインのパージデモ ===")
print("Mathドメインでのみ使用され、他(Text, Code)と共用されていないシナプスを検索します...")

# ユーティリティを用いて対象シナプスを特定
unshared_synapses = find_unshared_synapses(
    model=multi_model, 
    data_dict=data, 
    target_domain="math", 
    other_domains=["text", "code"], 
    get_batch_func=dummy_get_batch,
    max_seq_len=32,
    threshold=0.01  # 1%以上の頻度で利用されていれば「使用している」と判定
)

print(f"\n抽出されたMath専用のシナプスインデックス: {unshared_synapses}")

*Der obige Index kann bei einem nicht trainierten Modell möglicherweise nicht gefunden werden, da er zufällig verteilt ist.

Übergeben Sie den extrahierten Index (oder einen geeigneten Index für Demozwecke, falls er nicht gefunden wird) an „clear_synapses“, um die entsprechende Synapse auf Null zu setzen.

In [ ]:
if len(unshared_synapses) > 0:
    target_idx = unshared_synapses[0]
else:
    print("※未学習モデルのため条件に完全合致するものがありませんでした。デモとしてシナプス 3 を対象にします。")
    unshared_synapses = [3]
    target_idx = 3

# クリア前の重みのノルムを確認
pre_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
pre_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア前] シナプス {target_idx} の Router埋め込みノルム: {pre_emb_norm:.4f}, W1ノルム: {pre_w1_norm:.4f}")

# ゼロクリア実行
multi_model.clear_synapses(unshared_synapses)
print("\nゼロクリアを実行しました。\n")

# クリア後の重みのノルムを確認
post_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
post_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア後] シナプス {target_idx} の Router埋め込みノルム: {post_emb_norm:.4f}, W1ノルム: {post_w1_norm:.4f}")

### Abschluss
„clear_synapses“ löscht die Gewichtung des angegebenen Index auf „0,0“, ohne ihn physisch zu entfernen.
Dies verhindert, dass die Indizes (IDs) anderer Synapsen driften und unnötige Berechnungspfade vollständig deaktiviert werden, während gleichzeitig die Kompatibilität mit bestehenden Metadatenmasken gewahrt bleibt. Es ist möglich, den Index später mit einem neuen Plugin zu überschreiben (Hot-Swap), wenn er ein leerer Steckplatz wird.